# Section 1: Structured Standardized Operating Procedures (SOPs)

This section implements **Standardized Operating Procedures** for a MetaGPT-inspired multi-agent software development pipeline.

**Key Idea:** Each agent follows a **fixed, sequential procedure** — a rigid chain where outputs flow directly from one role to the next via function arguments. The SOP defines:
- **Who** does what (role assignment)
- **In what order** (sequential stages)
- **With what output format** (structured JSON schemas)

This mirrors MetaGPT's SOP mechanism where predefined workflows ensure consistency and quality.

---

### SOP Pipeline Flow

```
User Requirement
      │
      ▼
┌─────────────────┐
│ Product Manager  │ → PRD Document
└────────┬────────┘
         ▼
┌─────────────────┐
│   Architect      │ → System Design
└────────┬────────┘
         ▼
┌─────────────────┐
│ Project Manager  │ → Task List
└────────┬────────┘
         ▼
┌─────────────────┐
│   Engineer       │ → Code Files
└────────┬────────┘
         ▼
┌─────────────────┐
│  QA Engineer     │ → Tests + Review
└─────────────────┘
```

---
## Cell 1 — Imports and API Key

In [ ]:
# ============================================================
# CELL 1: Imports and API Key
# ============================================================

import json
from dataclasses import dataclass, field

import requests

API_KEY = "" #should add the API key here

print("Imports ready.")

Imports ready.


---
## Cell 2 — Define Output Schemas
Structured dataclasses that enforce a **standardized output format** for each role — a core part of the SOP.

In [2]:
# ============================================================
# CELL 2: Define Output Schemas
# ============================================================

@dataclass
class PRDDocument:
    original_requirements: str = ""
    product_goals: list = field(default_factory=list)
    user_stories: list = field(default_factory=list)
    competitive_analysis: list = field(default_factory=list)
    requirement_analysis: str = ""
    requirement_pool: list = field(default_factory=list)
    ui_design_draft: str = ""

@dataclass
class SystemDesign:
    implementation_approach: str = ""
    file_list: list = field(default_factory=list)
    data_structures: str = ""
    interface_definitions: str = ""
    program_call_flow: str = ""

@dataclass
class TaskList:
    required_packages: list = field(default_factory=list)
    task_list: list = field(default_factory=list)
    shared_knowledge: str = ""
    logic_analysis: list = field(default_factory=list)

@dataclass
class CodeOutput:
    filename: str = ""
    code: str = ""
    dependencies: list = field(default_factory=list)

@dataclass
class TestOutput:
    test_filename: str = ""
    test_code: str = ""
    review_notes: str = ""

print("Schemas defined.")

Schemas defined.


---
## Cell 3 — Define Prompt Templates
Each role has a **standardized prompt template** — part of the SOP that ensures consistent, structured LLM outputs.

In [3]:
# ============================================================
# CELL 3: Define Prompt Templates
# ============================================================

PRODUCT_MANAGER_PROMPT = """
You are a Product Manager. Given the user requirement below,
produce a Product Requirement Document in EXACTLY this JSON format:

{{
    "original_requirements": "<restate the requirement>",
    "product_goals": ["goal1", "goal2", "goal3"],
    "user_stories": ["story1", "story2", "story3"],
    "competitive_analysis": ["competitor1: analysis", "competitor2: analysis"],
    "requirement_analysis": "<detailed analysis>",
    "requirement_pool": [
        ["<requirement description>", "P0"],
        ["<requirement description>", "P1"]
    ],
    "ui_design_draft": "<description of UI layout>"
}}

User Requirement: {user_requirement}

Respond ONLY with valid JSON. No other text.
"""

ARCHITECT_PROMPT = """
You are a Software Architect. Given the PRD below, produce a
System Design document in EXACTLY this JSON format:

{{
    "implementation_approach": "<describe tech stack and approach>",
    "file_list": ["file1.py", "file2.py"],
    "data_structures": "<class definitions with attributes and methods>",
    "interface_definitions": "<interface specs between modules>",
    "program_call_flow": "<describe the sequence of function calls>"
}}

Product Requirement Document:
{prd}

Respond ONLY with valid JSON. No other text.
"""

PROJECT_MANAGER_PROMPT = """
You are a Project Manager. Given the System Design below, break it
down into tasks in EXACTLY this JSON format:

{{
    "required_packages": ["package1", "package2"],
    "task_list": ["file1.py", "file2.py"],
    "shared_knowledge": "<common knowledge all engineers need>",
    "logic_analysis": [
        ["file1.py", "description of what this file does"],
        ["file2.py", "description of what this file does"]
    ]
}}

System Design:
{system_design}

Respond ONLY with valid JSON. No other text.
"""

ENGINEER_PROMPT = """
You are a Software Engineer. Given the task, system design, and PRD below,
write the complete code for the specified file.

Respond in EXACTLY this JSON format:
{{
    "filename": "<filename>",
    "code": "<complete python code>",
    "dependencies": ["dep1", "dep2"]
}}

Task: Write {filename}
File Description: {file_description}

System Design:
{system_design}

PRD:
{prd}

Previously Written Code:
{existing_code}

Respond ONLY with valid JSON. No other text.
"""

QA_ENGINEER_PROMPT = """
You are a QA Engineer. Given the code below, write unit tests
and review the code for issues.

Respond in EXACTLY this JSON format:
{{
    "test_filename": "test_{filename}",
    "test_code": "<complete test code>",
    "review_notes": "<any issues found>"
}}

Code to Test:
{code}

PRD:
{prd}

Respond ONLY with valid JSON. No other text.
"""

print("Prompts defined.")

Prompts defined.


---
## Cell 4 — LLM Client
HTTP-based client for calling the OpenAI API.

In [4]:
# ============================================================
# CELL 4: LLM Client using requests (no openai library needed)
# ============================================================

class LLMClient:
    def __init__(self, api_key: str):
        self.api_key = api_key
        self.url = "https://api.openai.com/v1/chat/completions"

    def call(self, prompt: str) -> str:
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        data = {
            "model": "gpt-3.5-turbo",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2
        }
        response = requests.post(self.url, headers=headers, json=data)
        response.raise_for_status()
        return response.json()["choices"][0]["message"]["content"]

print("LLM Client ready.")

LLM Client ready.


---
## Cell 5 — Base Agent with Validation

The **BaseAgent** enforces the SOP pattern:
1. Build a standardized prompt
2. Call the LLM
3. Parse and validate the structured output
4. Retry on failure (up to 3 attempts)

Every role agent inherits this procedure — ensuring **consistency across all stages**.

In [5]:
# ============================================================
# CELL 5: Base Agent with Validation
# ============================================================

class BaseAgent:
    def __init__(self, llm: LLMClient):
        self.llm = llm
        self.role = "Base"
        self.output_schema = None

    def _build_prompt(self, **kwargs) -> str:
        raise NotImplementedError

    def _parse_response(self, response: str) -> dict:
        cleaned = response.strip()
        if cleaned.startswith("```json"):
            cleaned = cleaned[7:]
        if cleaned.startswith("```"):
            cleaned = cleaned[3:]
        if cleaned.endswith("```"):
            cleaned = cleaned[:-3]
        return json.loads(cleaned.strip())

    def _validate(self, parsed: dict) -> bool:
        """Validate output — checks that it's a dict. Missing fields get defaults."""
        if self.output_schema is None:
            return True
        if not isinstance(parsed, dict):
            print(f"    [{self.role}] Response is not a dict")
            return False
        return True

    def _to_dataclass(self, parsed: dict):
        """Convert to dataclass, letting defaults handle missing fields."""
        if self.output_schema is None:
            return parsed
        valid_fields = set(self.output_schema.__dataclass_fields__.keys())
        filtered = {k: v for k, v in parsed.items() if k in valid_fields}
        return self.output_schema(**filtered)

    def run(self, max_retries: int = 3, **kwargs):
        prompt = self._build_prompt(**kwargs)
        for attempt in range(max_retries):
            print(f"  [{self.role}] Attempt {attempt + 1}/{max_retries}...")
            try:
                response = self.llm.call(prompt)
                parsed = self._parse_response(response)
                if self._validate(parsed):
                    print(f"  [{self.role}] ✅ Success!")
                    return self._to_dataclass(parsed)
                else:
                    print(f"  [{self.role}] Validation failed, retrying...")
            except json.JSONDecodeError as e:
                print(f"  [{self.role}] JSON error: {e}, retrying...")
            except Exception as e:
                print(f"  [{self.role}] Error: {e}, retrying...")
        raise RuntimeError(f"[{self.role}] Failed after {max_retries} attempts")

print("Base Agent ready.")

Base Agent ready.


---
## Cell 6 — All Role Agents

Each role inherits from `BaseAgent` and implements its **specific SOP step**:
- Defines which prompt template to use
- Defines which output schema to validate against
- Receives input directly from the previous stage (function arguments)

In [6]:
# ============================================================
# CELL 6: All Role Agents
# ============================================================

class ProductManager(BaseAgent):
    def __init__(self, llm):
        super().__init__(llm)
        self.role = "Product Manager"
        self.output_schema = PRDDocument

    def _build_prompt(self, user_requirement="", **kw):
        return PRODUCT_MANAGER_PROMPT.format(user_requirement=user_requirement)


class Architect(BaseAgent):
    def __init__(self, llm):
        super().__init__(llm)
        self.role = "Architect"
        self.output_schema = SystemDesign

    def _build_prompt(self, prd=None, **kw):
        return ARCHITECT_PROMPT.format(prd=json.dumps(prd.__dict__, indent=2))


class ProjectManager(BaseAgent):
    def __init__(self, llm):
        super().__init__(llm)
        self.role = "Project Manager"
        self.output_schema = TaskList

    def _build_prompt(self, system_design=None, **kw):
        return PROJECT_MANAGER_PROMPT.format(
            system_design=json.dumps(system_design.__dict__, indent=2)
        )


class Engineer(BaseAgent):
    def __init__(self, llm):
        super().__init__(llm)
        self.role = "Engineer"
        self.output_schema = CodeOutput

    def _build_prompt(self, filename="", file_description="",
                      system_design=None, prd=None, existing_code=None, **kw):
        return ENGINEER_PROMPT.format(
            filename=filename,
            file_description=file_description,
            system_design=json.dumps(system_design.__dict__, indent=2),
            prd=json.dumps(prd.__dict__, indent=2),
            existing_code=json.dumps(existing_code or {}, indent=2)
        )


class QAEngineer(BaseAgent):
    def __init__(self, llm):
        super().__init__(llm)
        self.role = "QA Engineer"
        self.output_schema = TestOutput

    def _build_prompt(self, code=None, prd=None, **kw):
        return QA_ENGINEER_PROMPT.format(
            filename=code.filename,
            code=code.code,
            prd=json.dumps(prd.__dict__, indent=2)
        )

print("All agents defined.")

All agents defined.


---
## Cell 7 — SOP Pipeline

The **SOPPipeline** enforces the standardized operating procedure:
- Each stage runs **sequentially** in a fixed order
- Output from one stage is **directly passed** as input to the next
- No deviation from the defined workflow

```
Stage 1: PM(requirement) → PRD
Stage 2: Architect(PRD) → Design
Stage 3: ProjMgr(Design) → Tasks
Stage 4: Engineer(Tasks + Design + PRD) → Code (for each file)
Stage 5: QA(Code + PRD) → Tests (for each file)
```

In [7]:
# ============================================================
# CELL 7: SOP Pipeline
# ============================================================

class SOPPipeline:
    """
    Standardized Operating Procedure Pipeline.

    Enforces a fixed, sequential workflow:
    PM → Architect → Project Manager → Engineer → QA

    Each agent receives its input directly via function arguments
    from the previous stage — no message bus, no events.
    """

    def __init__(self, llm: LLMClient):
        self.pm = ProductManager(llm)
        self.architect = Architect(llm)
        self.proj_mgr = ProjectManager(llm)
        self.engineer = Engineer(llm)
        self.qa = QAEngineer(llm)

    def run(self, user_requirement: str) -> dict:

        # Stage 1: Product Manager → PRD
        print("=" * 50)
        print("STAGE 1: Product Manager → PRD")
        print("=" * 50)
        prd = self.pm.run(user_requirement=user_requirement)

        # Stage 2: Architect → System Design
        print("\n" + "=" * 50)
        print("STAGE 2: Architect → System Design")
        print("=" * 50)
        design = self.architect.run(prd=prd)

        # Stage 3: Project Manager → Task List
        print("\n" + "=" * 50)
        print("STAGE 3: Project Manager → Task List")
        print("=" * 50)
        tasks = self.proj_mgr.run(system_design=design)

        # Stage 4: Engineer → Code (for each file)
        print("\n" + "=" * 50)
        print("STAGE 4: Engineer → Code")
        print("=" * 50)
        code_files = {}
        for filename, desc in tasks.logic_analysis:
            print(f"\n--- Writing: {filename} ---")
            code = self.engineer.run(
                filename=filename,
                file_description=desc,
                system_design=design,
                prd=prd,
                existing_code={k: v.code for k, v in code_files.items()}
            )
            code_files[filename] = code

        # Stage 5: QA Engineer → Tests
        print("\n" + "=" * 50)
        print("STAGE 5: QA Engineer → Tests")
        print("=" * 50)
        test_files = {}
        for fname, code in code_files.items():
            print(f"\n--- Testing: {fname} ---")
            test = self.qa.run(code=code, prd=prd)
            test_files[fname] = test

        return {
            "prd": prd,
            "system_design": design,
            "task_list": tasks,
            "code": code_files,
            "tests": test_files
        }

print("SOP Pipeline ready.")

SOP Pipeline ready.


---
## Cell 8 — Run the SOP Pipeline

In [9]:
# ============================================================
# CELL 8: RUN IT
# ============================================================

llm = LLMClient(api_key=API_KEY)
pipeline = SOPPipeline(llm)

result = pipeline.run("Create a classic and simple Flappy Bird game")

STAGE 1: Product Manager → PRD
  [Product Manager] Attempt 1/3...
  [Product Manager] ✅ Success!

STAGE 2: Architect → System Design
  [Architect] Attempt 1/3...
  [Architect] ✅ Success!

STAGE 3: Project Manager → Task List
  [Project Manager] Attempt 1/3...
  [Project Manager] ✅ Success!

STAGE 4: Engineer → Code

--- Writing: game.js ---
  [Engineer] Attempt 1/3...
  [Engineer] ✅ Success!

--- Writing: styles.css ---
  [Engineer] Attempt 1/3...
  [Engineer] ✅ Success!

STAGE 5: QA Engineer → Tests

--- Testing: game.js ---
  [QA Engineer] Attempt 1/3...
  [QA Engineer] ✅ Success!

--- Testing: styles.css ---
  [QA Engineer] Attempt 1/3...
  [QA Engineer] ✅ Success!


---
## Cell 9 — View PRD Output

In [10]:
# ============================================================
# CELL 9: View PRD Output
# ============================================================

print("PRODUCT GOALS:")
for g in result["prd"].product_goals:
    print(f"  - {g}")

print("\nUSER STORIES:")
for s in result["prd"].user_stories:
    print(f"  - {s}")

print("\nREQUIREMENT POOL:")
for req, priority in result["prd"].requirement_pool:
    print(f"  [{priority}] {req}")

PRODUCT GOALS:
  - Engaging gameplay
  - Easy to learn
  - Addictive experience

USER STORIES:
  - As a player, I want to control a bird to avoid obstacles and earn points
  - As a player, I want to challenge my friends' high scores
  - As a player, I want to feel a sense of accomplishment when I progress in the game

REQUIREMENT POOL:
  [P0] Implement tap control for bird's movement
  [P1] Add scoring system based on distance traveled


---
## Cell 10 — View System Design

In [11]:
# ============================================================
# CELL 10: View System Design
# ============================================================

print("IMPLEMENTATION APPROACH:")
print(f"  {result['system_design'].implementation_approach}")

print("\nFILE LIST:")
for f in result["system_design"].file_list:
    print(f"  - {f}")

print("\nDATA STRUCTURES:")
print(f"  {result['system_design'].data_structures}")

IMPLEMENTATION APPROACH:
  The game will be implemented using HTML5, CSS, and JavaScript. The game logic will be handled in JavaScript, while CSS will be used for styling the UI. The game will be designed to run in a web browser.

FILE LIST:
  - game.js
  - styles.css

DATA STRUCTURES:
  Bird class with attributes such as position and velocity, Obstacle class with attributes such as position and size. Methods for collision detection and game loop.


---
## Cell 11 — View Task Breakdown

In [12]:
# ============================================================
# CELL 11: View Task Breakdown
# ============================================================

print("TASKS:")
for fname, desc in result["task_list"].logic_analysis:
    print(f"  {fname}: {desc}")

print("\nREQUIRED PACKAGES:")
for pkg in result["task_list"].required_packages:
    print(f"  - {pkg}")

TASKS:
  game.js: Handles game logic such as user input, collision detection, scoring system, and game over screen
  styles.css: Responsible for styling the UI of the game

REQUIRED PACKAGES:
  - HTML5
  - CSS
  - JavaScript


---
## Cell 12 — View Generated Code

In [13]:
# ============================================================
# CELL 12: View Generated Code
# ============================================================

for fname, code_obj in result["code"].items():
    print(f"\n{'=' * 50}")
    print(f"FILE: {fname}")
    print(f"{'=' * 50}")
    print(code_obj.code)


FILE: game.js
const Bird = class {
    constructor() {
        this.position = { x: 0, y: 0 };
        this.velocity = { x: 0, y: 0 };
    }
}

const Obstacle = class {
    constructor() {
        this.position = { x: 0, y: 0 };
        this.size = { width: 0, height: 0 };
    }
}

function handleUserInput() {
    // code for handling user input
}

function collisionDetection() {
    // code for collision detection
}

function scoringSystem() {
    // code for updating scoring system
}

function gameOverScreen() {
    // code for displaying game over screen
}


FILE: styles.css



---
## Cell 13 — View Tests and Reviews

In [14]:
# ============================================================
# CELL 13: View Tests and Reviews
# ============================================================

for fname, test_obj in result["tests"].items():
    print(f"\n{'=' * 50}")
    print(f"TEST FOR: {fname}")
    print(f"{'=' * 50}")
    print(test_obj.test_code)
    print(f"\nReview: {test_obj.review_notes}")


TEST FOR: game.js
const assert = require('assert');
const { Bird, Obstacle, handleUserInput, collisionDetection, scoringSystem, gameOverScreen } = require('./game');

describe('Bird', () => {
    it('should create a bird object with default position and velocity', () => {
        const bird = new Bird();
        assert.deepStrictEqual(bird.position, { x: 0, y: 0 });
        assert.deepStrictEqual(bird.velocity, { x: 0, y: 0 });
    });
});

describe('Obstacle', () => {
    it('should create an obstacle object with default position and size', () => {
        const obstacle = new Obstacle();
        assert.deepStrictEqual(obstacle.position, { x: 0, y: 0 });
        assert.deepStrictEqual(obstacle.size, { width: 0, height: 0 });
    });
});

Review: No issues found

TEST FOR: styles.css
describe('Flappy Bird Game', () => {
  it('should have tap control for bird movement', () => {
    // Test implementation
  });

  it('should have a scoring system based on distance traveled', () => {
   